# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [7]:
# Add code here 🔧
# Get the Data in Python using pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
hotels.shape

# 1. Load the dataset
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv'
hotels = pd.read_csv(url)

# 2. Separate into features (X) and target (y)
# We drop 'reservation_status' and 'reservation_status_date' as they directly leak whether a booking was canceled
features_to_drop = ['is_canceled', 'reservation_status', 'reservation_status_date']
X = hotels.drop(columns=features_to_drop, errors='ignore')
y = hotels['is_canceled']

# 3. Train-Test Split (70/30) BEFORE any preprocessing or feature transformations
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# 4. Impute missing values
# Identify numeric and categorical columns
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Impute missing numerical values using training median
for col in num_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# Impute missing categorical values using training mode (or placeholder)
for col in cat_cols:
    mode_val = X_train[col].mode()[0] if not X_train[col].mode().empty else 'Unknown'
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

# 5. Encode Categorical Variables
# Using pandas get_dummies to one-hot encode. Align train and test dummy columns to ensure identical feature sets.
X_train_encoded = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Align columns between training and test sets to handle differences in category distributions
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Check shapes to verify everything is set up correctly
print("X_train_encoded shape:", X_train_encoded.shape)
print("X_test_encoded shape:", X_test_encoded.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train_encoded shape: (83573, 235)
X_test_encoded shape: (35817, 235)
y_train shape: (83573,)
y_test shape: (35817,)


### ✍️ Your Response: 🔧
1. There are 119,390 rows and 235 columns in this data set

2. The dataset includes numerical features, such as lead time and average daily rate, and categorical features, such as hotel type, meal plan, and country.

3. I separated the target from the features, split the data into training and testing sets, handled missing values, and encoded categorical variables.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [ ]:
# Add code here 🔧


In [8]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Naïve Bayes Classifier
nb_model = GaussianNB()

# Fit the model on the training data
nb_model.fit(X_train_encoded, y_train)

# Make predictions on the test set
y_pred = nb_model.predict(X_test_encoded)

# Print Evaluation Metrics
print("=== Naïve Bayes Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Naïve Bayes Classification Report ===
              precision    recall  f1-score   support

           0       0.87      0.32      0.46     22550
           1       0.44      0.92      0.60     13267

    accuracy                           0.54     35817
   macro avg       0.65      0.62      0.53     35817
weighted avg       0.71      0.54      0.51     35817

=== Confusion Matrix ===
[[ 7113 15437]
 [ 1096 12171]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The model had 54% accuracy and detected 92% of cancellations, so the F1-score is the best metric because the classes are imbalanced

2. The model could be used to flag likely cancellations so the hotel can send reminders, adjust staffing, or offer rooms to other guests

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [ ]:
# Add code here 🔧

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale the data using StandardScaler
scaler = StandardScaler()

# Fit the scaler ONLY on the training data, then transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# 2. Train an SVM classifier (Linear kernel)
# We use LinearSVC because SVC(kernel='linear') is extremely slow for >80k rows.
svm_model = LinearSVC(random_state=42, max_iter=2000, dual=False)
svm_model.fit(X_train_scaled, y_train)

# 3. Make predictions on the test set
y_pred_svm = svm_model.predict(X_test_scaled)

# 4. Evaluate with classification metrics
print("=== SVM (Linear) Classification Report ===")
print(classification_report(y_test, y_pred_svm))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_svm))

=== SVM (Linear) Classification Report ===
              precision    recall  f1-score   support

           0       0.82      0.91      0.86     22550
           1       0.81      0.65      0.72     13267

    accuracy                           0.82     35817
   macro avg       0.81      0.78      0.79     35817
weighted avg       0.81      0.82      0.81     35817

=== Confusion Matrix ===
[[20571  1979]
 [ 4646  8621]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The SVM performed well with 82% accuracy, and the F1-score is the best metric because it considers both precision and recall

2. SVM can provide better insights when the hotel needs to predict cancellations using many customer and booking features at once

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [ ]:
# Add code here 🔧

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Create a true validation split from the training data (e.g., 15% of X_train_scaled)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# 2. Build the MLPClassifier model
# We choose a simple 2-layer architecture: (64, 32)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    verbose=True
)

# 3. Fit the model
# early_stopping=True natively uses a validation split for convergence monitoring
mlp_model.fit(X_tr, y_tr)

# 4. Make predictions on the test set
y_pred_mlp = mlp_model.predict(X_test_scaled)

# 5. Evaluate Accuracy and Performance
print("\n=== Neural Network (MLP) Classification Report ===")
print(classification_report(y_test, y_pred_mlp))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_mlp))

Iteration 1, loss = 0.42688900
Validation score: 0.832395
Iteration 2, loss = 0.34315248
Validation score: 0.839058
Iteration 3, loss = 0.32645491
Validation score: 0.845252
Iteration 4, loss = 0.31579266
Validation score: 0.844032
Iteration 5, loss = 0.30815197
Validation score: 0.849099
Iteration 6, loss = 0.30122693
Validation score: 0.849568
Iteration 7, loss = 0.29792640
Validation score: 0.851445
Iteration 8, loss = 0.29253921
Validation score: 0.854167
Iteration 9, loss = 0.28822912
Validation score: 0.853697
Iteration 10, loss = 0.28430611
Validation score: 0.855762
Iteration 11, loss = 0.28101933
Validation score: 0.858671
Iteration 12, loss = 0.27740001
Validation score: 0.857076
Iteration 13, loss = 0.27520835
Validation score: 0.859140
Iteration 14, loss = 0.27227977
Validation score: 0.859891
Iteration 15, loss = 0.26949696
Validation score: 0.860642
Iteration 16, loss = 0.26839071
Validation score: 0.859422
Iteration 17, loss = 0.26582110
Validation score: 0.859610
Iterat

### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The neural network performed best with 86% accuracy, compared with 82% for SVM and 54% for Naïve Bayes.

2. The business could use this model because it is the most accurate, but it may be harder to explain its predictions.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [ ]:
# Add code here 🔧

In [11]:
from sklearn.metrics import accuracy_score

# Calculate accuracy scores for each model on the test set
nb_accuracy = accuracy_score(y_test, y_pred)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
mlp_accuracy = accuracy_score(y_test, y_pred_mlp)

# Print the results in a formatted comparison table
print("====================================================")
print("       Model Accuracy Comparison on Test Set        ")
print("====================================================")
print(f" 1. Naïve Bayes Accuracy  : {nb_accuracy:.2%}")
print(f" 2. Linear SVM Accuracy   : {svm_accuracy:.2%}")
print(f" 3. Neural Network (MLP)  : {mlp_accuracy:.2%}")
print("====================================================")

       Model Accuracy Comparison on Test Set        
 1. Naïve Bayes Accuracy  : 53.84%
 2. Linear SVM Accuracy   : 81.50%
 3. Neural Network (MLP)  : 85.81%


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The neural network had the best accuracy (85.81%), while Naïve Bayes was likely fastest and easiest to use, and SVM was more interpretable

2. I would recommend the neural network because it had the highest accuracy, although the SVM may be preferred if explaining predictions is important

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend implementing the neural network because it achieved the highest accuracy (85.81%) for predicting hotel booking cancellations. This model can help management identify likely cancellations early and adjust staffing, pricing, and room availability. A limitation is that neural networks are harder to explain and may still incorrectly label some bookings as cancellations. Adding customer booking history, cancellation reasons, local events, and competitor pricing could improve future predictions.


2. This relates to my learning outcome because I used Python to clean and prepare hotel booking data and translated cancellations into an actionable business problem using analytics.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [12]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 332410 bytes to assignment_12_bayes_svm_neural.html
